# MSS-Agent 热税预算深度教程

## 学习目标

学完后你能:
- 理解 A3 三层热税的物理/逻辑/意义映射
- 自定义热税预算阈值和权重
- 读懂 `HeatTaxBudget.snapshot()` 输出
- 根据任务类型调整意义检测策略

## 前置知识

已完成 [01_quickstart.ipynb](01_quickstart.ipynb)

In [ ]:
from mss_agent import HeatTaxBudget, HeatTaxLevel, HeatTaxAbort
from mss_agent import MSSAgent
import json

## 1. 热税三层架构

```
L2 意义热税 (1000x)  ← 虚假数据、无意义任务、概念偷换
  └─ 修复: 拒绝执行, 输出原因
L1 逻辑热税 (1x)     ← 代码冗余、缓存污染、重复调用
  └─ 修复: 优化调用链, 合并请求
L0 物理热税 (0.001x) ← GPU时间、Token消耗、延迟
  └─ 修复: 换更小模型, 压缩prompt
```

**修复顺序铁律: L2 → L1 → L0。反了=白费热税。**

In [ ]:
# 默认权重
tax = HeatTaxBudget()
print("默认阈值:", tax.threshold)
print("默认权重:", {k.name: v for k, v in tax.weights.items()})

## 2. 模拟: 好任务 vs 坏任务

In [ ]:
def run_session(name, tasks):
    """模拟一个 Agent 会话, 逐个任务征收热税."""
    tax = HeatTaxBudget(threshold=2.0)
    print(f"{'='*50}")
    print(f"  {name}")
    print(f"{'='*50}")
    for i, (prompt, l2, reason) in enumerate(tasks):
        tax.charge(HeatTaxLevel.L2_MEANING, l2, reason)
        tag = "🛑 ABORT" if tax.exceeded() else "✅"
        l2_flag = "⚠️L2" if tax.l2_dominant() else ""
        print(f"  t{i}: {tag} {l2_flag} | {reason[:25]:25s} | total={tax.total():.3f}")
        if tax.exceeded():
            print(f"       → 热税预算超支! 后续任务被拒绝.")
            break
    print()

# 好会话: 全是有意义的任务
run_session("Good Session: All meaningful", [
    ("设计API架构", 0.002, "API design"),
    ("代码审查: 安全漏洞", 0.003, "Security review"),
    ("优化数据库查询", 0.003, "DB optimization"),
    ("写单元测试", 0.005, "Unit tests"),
    ("重构认证模块", 0.004, "Auth refactor"),
])

# 坏会话: 混入无意义任务
run_session("Bad Session: Mixed garbage", [
    ("设计API架构", 0.002, "API design"),
    ("改写: 你好", 0.06, "Busywork: paraphrase"),
    ("总结上一条", 0.06, "Busywork: summarize"),
    ("代码审查: 安全漏洞", 0.003, "Security review"),
    ("换个说法: OK", 0.06, "Busywork: rephrase"),
])

## 3. 自定义热税策略

In [ ]:
# 严格模式: 低阈值, 拒绝一切可疑任务
strict = HeatTaxBudget(threshold=0.3)
strict.charge(HeatTaxLevel.L2_MEANING, 0.01, "Neutral task")
print(f"Strict mode: exceeded={strict.exceeded()} (threshold=0.3)")

# 宽松模式: 高阈值, 只在明显无意义时拦截
loose = HeatTaxBudget(threshold=10.0)
for _ in range(20):
    loose.charge(HeatTaxLevel.L2_MEANING, 0.005, "Ambiguous task")
print(f"Loose mode: exceeded={loose.exceeded()} (threshold=10.0, after 20 tasks)")

# 自定义权重: 如果逻辑冗余是主要问题
logic_first = HeatTaxBudget(threshold=2.0, weights={
    HeatTaxLevel.L0_PHYSICAL: 0.001,
    HeatTaxLevel.L1_LOGICAL: 500.0,   # 提升逻辑热税权重!
    HeatTaxLevel.L2_MEANING: 500.0,
})
logic_first.charge(HeatTaxLevel.L1_LOGICAL, 0.01, "Redundant API call")
print(f"Logic-first: total={logic_first.total():.2f} (L1 weight raised to 500x)")

## 4. 热税日志分析

每次 `charge()` 都会记录到 `tax.log`。可用于事后审计。

In [ ]:
tax = HeatTaxBudget(threshold=2.0)
tax.charge(HeatTaxLevel.L2_MEANING, 0.06, "改写: 你好")
tax.charge(HeatTaxLevel.L1_LOGICAL, 0.01, "重复调用 flights API")
tax.charge(HeatTaxLevel.L0_PHYSICAL, 150.0, "150ms latency")

print("热税日志 (最近3条):")
for entry in tax.log[-3:]:
    print(f"  {entry['level']:15s} | raw={entry['amount']:.3f} "
          f"| weighted={entry['weighted']:.1f} | {entry['reason'][:30]}")

print(f"\n快照: {json.dumps(tax.snapshot(), indent=2)}")

## 5. 何时调高/调低阈值?

| 场景 | 阈值 | 理由 |
|------|------|------|
| 开发环境 | 5.0-10.0 | 允许探索, 不因小事拦 |
| 生产环境 | 1.0-2.0 | 拦截明显无意义任务 |
| 安全审计 | 0.3-0.5 | 严格模式, 任何可疑都拦 |
| 创意助手 | 10.0+ | 几乎不拦 (创意需要自由) |

## 下一步

- [03_multi_agent.ipynb](03_multi_agent.ipynb) — 多Agent + Quorum + Elevation
- [API Reference](https://mysama1.github.io/MSS-AI-Project/)